# PCD BBox Crop Batch (Google Colab)

指定したバウンディングボックス(BBox)の範囲内のみを残してPCDを切り取り、
入力ディレクトリ配下の全PCDを **同じディレクトリ階層を保ったまま** 出力ディレクトリに保存します。

## 使い方
1. **環境準備** セルで `open3d` をインストール
2. **Google Drive マウント** セルでデータを置いたDriveをマウント
3. **パラメータ設定** セルで `INPUT_DIR` / `OUTPUT_DIR` / BBox を設定
    - BBox は直接指定するか、`bbox_config.json` から読み込み可能
4. **バッチ処理実行** セルを実行

## 入出力例
```
INPUT_DIR/
├── session_A/
│   ├── 000001.pcd
│   └── 000002.pcd
└── session_B/
    └── sub/
        └── 000001.pcd

OUTPUT_DIR/
├── session_A/
│   ├── 000001.pcd     ← BBox内のみ
│   └── 000002.pcd
└── session_B/
    └── sub/
        └── 000001.pcd
```

## 1. 環境準備

Google Colab上で `open3d` をインストールします。
ローカルJupyterで既に入っている場合はスキップ可能。

In [ ]:
!pip install -q open3d

## 2. Google Drive マウント (Colab使用時のみ)

ローカル環境では実行不要。Colabで実行する場合のみ実行してください。

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive をマウントしました: /content/drive')
except ImportError:
    print('Colab環境ではありません。Driveマウントをスキップします。')

## 3. ライブラリ・関数定義

In [ ]:
import json
import os
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import open3d as o3d

PARAM_NAMES = ['xmin', 'xmax', 'ymin', 'ymax', 'zmin', 'zmax']


def load_bbox_config(config_path: str) -> Optional[Tuple[float, ...]]:
    """bbox_config.json から BBox を読み込む"""
    if not os.path.exists(config_path):
        print(f'設定ファイルが見つかりません: {config_path}')
        return None
    with open(config_path, 'r') as f:
        data = json.load(f)
    try:
        return tuple(float(data[name]) for name in PARAM_NAMES)
    except (KeyError, ValueError) as e:
        print(f'設定ファイルの読み込みに失敗しました: {e}')
        return None


def build_inside_mask(points: np.ndarray, region: Tuple[float, ...]) -> np.ndarray:
    """region 内に含まれる点のbool マスクを返す"""
    xmin, xmax, ymin, ymax, zmin, zmax = region
    return (
        (points[:, 0] >= xmin) & (points[:, 0] <= xmax) &
        (points[:, 1] >= ymin) & (points[:, 1] <= ymax) &
        (points[:, 2] >= zmin) & (points[:, 2] <= zmax)
    )


def filter_pcd_by_bbox(
    pcd: o3d.geometry.PointCloud, region: Tuple[float, ...]
) -> o3d.geometry.PointCloud:
    """BBox内の点群のみを残したPointCloudを返す (色・法線も保持)"""
    points = np.asarray(pcd.points)
    mask = build_inside_mask(points, region)
    return pcd.select_by_index(np.where(mask)[0])


def find_pcd_files(input_dir: str) -> List[Path]:
    """入力ディレクトリ配下の全PCDファイルを再帰的に列挙"""
    root = Path(input_dir)
    if not root.exists():
        raise FileNotFoundError(f'入力ディレクトリが見つかりません: {input_dir}')
    files = sorted(root.rglob('*.pcd'))
    return files


def process_directory(
    input_dir: str,
    output_dir: str,
    region: Tuple[float, ...],
    overwrite: bool = True,
    add_suffix: bool = False,
) -> None:
    """input_dir 配下の全PCDをBBoxで切り取り、output_dirに同階層で保存

    Args:
        input_dir: 入力ルートディレクトリ
        output_dir: 出力ルートディレクトリ
        region: (xmin,xmax,ymin,ymax,zmin,zmax)
        overwrite: 既存ファイルを上書きするか
        add_suffix: True なら出力ファイル名に '_filtered' を付与
    """
    input_root = Path(input_dir).resolve()
    output_root = Path(output_dir).resolve()
    pcd_files = find_pcd_files(str(input_root))

    if not pcd_files:
        print(f'PCDファイルが見つかりませんでした: {input_root}')
        return

    print(f'対象ファイル数: {len(pcd_files)}')
    print(f'BBox: X[{region[0]}, {region[1]}]  Y[{region[2]}, {region[3]}]  Z[{region[4]}, {region[5]}]')
    print(f'入力ルート: {input_root}')
    print(f'出力ルート: {output_root}')
    print('-' * 60)

    n_skipped = 0
    n_processed = 0
    n_empty = 0
    for i, src_path in enumerate(pcd_files, 1):
        rel_path = src_path.relative_to(input_root)
        if add_suffix:
            dst_path = output_root / rel_path.with_name(rel_path.stem + '_filtered' + rel_path.suffix)
        else:
            dst_path = output_root / rel_path
        dst_path.parent.mkdir(parents=True, exist_ok=True)

        if dst_path.exists() and not overwrite:
            print(f'  [{i}/{len(pcd_files)}] SKIP (already exists): {rel_path}')
            n_skipped += 1
            continue

        pcd = o3d.io.read_point_cloud(str(src_path))
        original_count = len(pcd.points)
        filtered_pcd = filter_pcd_by_bbox(pcd, region)
        filtered_count = len(filtered_pcd.points)

        o3d.io.write_point_cloud(str(dst_path), filtered_pcd)

        rate = 100.0 * filtered_count / original_count if original_count > 0 else 0.0
        marker = '  ' if filtered_count > 0 else '* '
        if filtered_count == 0:
            n_empty += 1
        print(f'{marker}[{i}/{len(pcd_files)}] {rel_path}: {original_count} -> {filtered_count} ({rate:.1f}%)')
        n_processed += 1

    print('-' * 60)
    print(f'完了: 処理 {n_processed} / スキップ {n_skipped} / 出力点数0 {n_empty}')
    print(f'出力先: {output_root}')

## 4. パラメータ設定

### 入出力ディレクトリ
- `INPUT_DIR`: PCDファイルが入っているルートディレクトリ (配下のサブディレクトリも再帰的に処理)
- `OUTPUT_DIR`: 切り取り後のPCDを保存するディレクトリ (入力と同じ階層構造で保存)

### BBox の指定方法 (2通り)
- **(a) 直接指定**: 下のセルで `REGION = (xmin, xmax, ymin, ymax, zmin, zmax)` を編集
- **(b) JSONから読み込み**: `BBOX_CONFIG_PATH` を設定して `load_bbox_config` を使う
    - `pcd_bbox_filter.py` で `S` キーを押した時に保存される `bbox_config.json` を流用可能

In [ ]:
# === 入出力ディレクトリ ===
# Colab + Google Drive 例:
#   INPUT_DIR  = '/content/drive/MyDrive/lidar_data/raw'
#   OUTPUT_DIR = '/content/drive/MyDrive/lidar_data/cropped'
# ローカル例:
#   INPUT_DIR  = '/path/to/input_root'
#   OUTPUT_DIR = '/path/to/output_root'
INPUT_DIR  = '/content/drive/MyDrive/lidar_data/raw'
OUTPUT_DIR = '/content/drive/MyDrive/lidar_data/cropped'

# === BBox 指定 ===
# (a) 直接指定する場合:
REGION: Tuple[float, ...] = (-5.0, 5.0, -3.0, 3.0, 0.0, 2.5)

# (b) JSONから読み込む場合: 下のコメントアウトを外す
# BBOX_CONFIG_PATH = '/content/drive/MyDrive/lidar_data/bbox_config.json'
# loaded = load_bbox_config(BBOX_CONFIG_PATH)
# if loaded is not None:
#     REGION = loaded

# === オプション ===
OVERWRITE  = True   # 既存出力ファイルを上書きするか
ADD_SUFFIX = False  # True にすると 'xxx.pcd' -> 'xxx_filtered.pcd' に

print('INPUT_DIR :', INPUT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('REGION    :', dict(zip(PARAM_NAMES, REGION)))

## 5. (オプション) 対象ファイルの事前確認

実行前にどのファイルが処理されるかを確認できます。

In [ ]:
files = find_pcd_files(INPUT_DIR)
print(f'検出ファイル数: {len(files)}')
for f in files[:20]:
    print(' ', f.relative_to(Path(INPUT_DIR).resolve()))
if len(files) > 20:
    print(f'  ... 他 {len(files) - 20} ファイル')

## 6. バッチ処理実行

In [ ]:
process_directory(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    region=REGION,
    overwrite=OVERWRITE,
    add_suffix=ADD_SUFFIX,
)

## 7. (オプション) 結果サンプル可視化

Colab はGUI可視化に制限があるため、点数の統計情報を表示します。
ローカルJupyterで実行する場合は `o3d.visualization.draw_geometries([pcd])` で可視化できます。

In [ ]:
out_files = sorted(Path(OUTPUT_DIR).rglob('*.pcd'))
print(f'出力ファイル数: {len(out_files)}')
for f in out_files[:5]:
    pcd = o3d.io.read_point_cloud(str(f))
    pts = np.asarray(pcd.points)
    print(f'  {f.relative_to(Path(OUTPUT_DIR).resolve())}: {len(pts)} 点')
    if len(pts) > 0:
        print(f'    X[{pts[:,0].min():.3f}, {pts[:,0].max():.3f}]  '
              f'Y[{pts[:,1].min():.3f}, {pts[:,1].max():.3f}]  '
              f'Z[{pts[:,2].min():.3f}, {pts[:,2].max():.3f}]')